# EDA — Washington DC Census Blocks & ICF Facilities

Exploratory analysis of the datasets produced in `data_preparation.ipynb`.

**Run kernel:** `arcgis_env` (has geopandas, matplotlib, numpy)

## Sections
1. Census Blocks — raw EDA
2. ICF Facilities — EDA
3. EDA on Joined Dataset (`blocks_with_income.shp`)
4. Disaggregated Block-Level Variable Distributions
5. Normalised Variable Maps

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_ROOT / 'data' / 'shapefiles'

print('Project root:', PROJECT_ROOT)
print('Data dir    :', DATA_DIR)

## 1. Census Blocks — Raw EDA

`Census_Blocks_in_2020.shp` — population only, no economic variables yet.

In [ ]:
blocks = gpd.read_file(DATA_DIR / 'Census_Blocks_in_2020.shp').to_crs('epsg:26985')
print(f'Shape  : {blocks.shape}')
print(f'CRS    : {blocks.crs}')
print(f'Columns: {list(blocks.columns)}\n')
blocks.head()

In [ ]:
print('=== Null counts ===')
print(blocks.isnull().sum())
print('\n=== Dtypes ===')
print(blocks.dtypes)
print('\n=== Geometry types ===')
print(blocks.geom_type.value_counts())

In [ ]:
blocks.describe()

In [ ]:
pop_col = [c for c in blocks.columns if 'pop' in c.lower() or 'popu' in c.lower()]
print('Population column(s) found:', pop_col)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

blocks.plot(column=pop_col[0] if pop_col else blocks.columns[1],
            cmap='YlOrRd', linewidth=0.2, edgecolor='grey',
            legend=True, ax=axes[0], missing_kwds={'color': 'lightgrey'})
axes[0].set_title('Population per Block (map)')
axes[0].set_axis_off()

if pop_col:
    blocks[pop_col[0]].hist(bins=60, ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].set_title('Population Distribution (histogram)')
    axes[1].set_xlabel('Population')
    axes[1].set_ylabel('Number of blocks')

    axes[2].boxplot(blocks[pop_col[0]].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[2].set_title('Population Boxplot')
    axes[2].set_ylabel('Population')

plt.tight_layout()
plt.show()

if pop_col:
    n_zero = (blocks[pop_col[0]] == 0).sum()
    pct_zero = n_zero / len(blocks) * 100
    print(f'\nZero-population blocks: {n_zero:,} ({pct_zero:.1f}%)')
    print('Note: Zero-pop blocks exist (parks, water, industrial) — handled in data_preparation.')

## 2. ICF Facilities — EDA

In [ ]:
icf = gpd.read_file(DATA_DIR / 'Intermediate_Care_Facilities.shp').to_crs('epsg:26985')
print(f'Shape  : {icf.shape}')
print(f'CRS    : {icf.crs}')
print(f'Columns: {list(icf.columns)}\n')
icf.head()

In [ ]:
print('=== Null counts ===')
print(icf.isnull().sum())
print('\n=== Basic stats ===')
icf.describe()

In [ ]:
bed_col = [c for c in icf.columns if 'bed' in c.lower()]
print('Bed column(s):', bed_col)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

blocks.plot(color='whitesmoke', linewidth=0.2, edgecolor='lightgrey', ax=axes[0])
icf.plot(ax=axes[0], color='crimson', markersize=40, zorder=5)
axes[0].set_title('ICF Facility Locations — Washington DC')
axes[0].set_axis_off()

if bed_col:
    icf[bed_col[0]].hist(bins=20, ax=axes[1], color='crimson', edgecolor='white', alpha=0.8)
    axes[1].set_title(f'ICF Bed Count Distribution ({bed_col[0]})')
    axes[1].set_xlabel('Beds')
    axes[1].set_ylabel('Number of facilities')
    print(f'\nBed count stats:\n{icf[bed_col[0]].describe()}')

plt.tight_layout()
plt.show()

## 3. EDA on Joined Dataset (`blocks_with_income.shp`)

Result of spatial join: each block inherits ACS variables from its containing census tract.

In [ ]:
blocks_eco = gpd.read_file(DATA_DIR / 'blocks_with_income.shp').to_crs('epsg:26985')
print(f'Shape  : {blocks_eco.shape}')
print(f'CRS    : {blocks_eco.crs}')
print(f'Columns: {list(blocks_eco.columns)}\n')
blocks_eco.head()

In [ ]:
raw_cols    = set(blocks.columns) - {'geometry'}
joined_cols = set(blocks_eco.columns) - {'geometry'}
new_cols    = joined_cols - raw_cols
print(f'Columns in raw blocks           : {sorted(raw_cols)}')
print(f'Columns added after spatial join: {sorted(new_cols)}')

In [ ]:
print('=== Shape ===')
print(blocks_eco.shape)
print('\n=== Null counts ===')
print(blocks_eco.isnull().sum())
print('\n=== Dtypes ===')
print(blocks_eco.dtypes)

In [ ]:
blocks_eco.describe()

In [ ]:
numeric_df = blocks_eco.select_dtypes(include=np.number)
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)

ticks = range(len(corr.columns))
ax.set_xticks(ticks); ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(ticks); ax.set_yticklabels(corr.columns, fontsize=9)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(corr.iloc[i, j]) > 0.6 else 'black')

ax.set_title('Correlation Matrix — Joined Block Dataset', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
eco_cols = list(numeric_df.columns)
n = len(eco_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flat

for ax, col in zip(axes, eco_cols):
    data = blocks_eco[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.2, label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.2, label=f'median={data.median():.1f}')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

for ax in list(axes)[n:]:
    ax.set_visible(False)

plt.suptitle('Distribution of All Variables — blocks_with_income.shp', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
def find_col(df, keywords):
    for kw in keywords:
        matches = [c for c in df.columns if kw.lower() in c.lower()]
        if matches:
            return matches[0]
    return None

col_income = find_col(blocks_eco, ['income', 'incomep', 'percap'])
col_hi     = find_col(blocks_eco, ['health', 'insur', 'hi'])
col_age    = find_col(blocks_eco, ['eighteen', '18to', '18_'])
col_blpop  = find_col(blocks_eco, ['bl_total', 'blpop', 'block_pop'])

plot_specs = [(col_income, 'Per-Capita Income (tract)', 'RdYlGn'),
              (col_hi,     'Health Insurance (tract)',  'Blues'),
              (col_age,    'Age 18-65 (tract)',         'Purples'),
              (col_blpop,  'Block Population',          'YlOrRd')]
plot_specs = [(c, t, cm) for c, t, cm in plot_specs if c is not None]

fig, axes = plt.subplots(1, len(plot_specs), figsize=(6 * len(plot_specs), 6))
if len(plot_specs) == 1:
    axes = [axes]

for ax, (col, title, cmap) in zip(axes, plot_specs):
    blocks_eco.plot(column=col, cmap=cmap, linewidth=0.2, edgecolor='grey',
                    legend=True, ax=ax, missing_kwds={'color': 'lightgrey'})
    icf.plot(ax=ax, color='black', markersize=7, zorder=5, label='ICF')
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()
    ax.legend(loc='lower right', fontsize=8)

plt.suptitle('Tract-Level Variables Joined to Blocks — Washington DC', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Disaggregated Block-Level Variable Distributions

Load the processed output from `data_preparation.ipynb` to inspect disaggregated variables.

In [ ]:
final = gpd.read_file(DATA_DIR / 'blocksandtract_economic_final.shp').to_crs('epsg:26985')
print(f'Shape  : {final.shape}')
print(f'Columns: {list(final.columns)}\n')
final.head()

In [ ]:
disagg_specs = [
    ('PerCapita', 'Per-Capita Income (block)', 'teal'),
    ('HI_block',  'Health Insurance (block)',  'steelblue'),
    ('age_18to6', 'Age 18-65 (block)',         'purple'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (hint, title, color) in zip(axes, disagg_specs):
    col = next((c for c in final.columns if c.startswith(hint[:7])), None)
    if col is None:
        ax.set_visible(False)
        continue
    final[col].dropna().hist(bins=50, ax=ax, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(final[col].mean(),   color='red',    linestyle='--', linewidth=1.2, label='mean')
    ax.axvline(final[col].median(), color='orange', linestyle='--', linewidth=1.2, label='median')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Disaggregated Block-Level Variable Distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Normalised Variable Maps

In [ ]:
norm_specs = [
    ('norm_tota', 'Block Population (normalised)',            'YlOrRd'),
    ('norm_inco', 'Per-Capita Income (normalised)',           'RdYlGn'),
    ('norm_heal', 'Health Insurance Coverage (normalised)',   'Blues'),
    ('norm_age_', 'Working-Age Population 18-65 (normalised)', 'Purples'),
]

resolved = [(next((c for c in final.columns if c.startswith(hint[:7])), None), title, cmap)
            for hint, title, cmap in norm_specs]
resolved = [(c, t, cm) for c, t, cm in resolved if c is not None]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (col, title, cmap) in zip(axes.flat, resolved):
    final.plot(column=col, cmap=cmap, linewidth=0.2, edgecolor='grey',
               legend=True, ax=ax, missing_kwds={'color': 'lightgrey'})
    icf.plot(ax=ax, color='black', markersize=6, zorder=5, label='ICF')
    ax.set_title(title, fontsize=12)
    ax.set_axis_off()
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('Washington DC — Final Normalised Block-Level Variables', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()